<div dir="rtl">

# جلسه ۲۴: Pandas ۲
## Pandas II

**بخش ۳: محاسبات علمی و کار با داده** — دوره‌ی پایتون برای علوم داده

مدرس: دکتر حسین حق‌بین

</div>

<div dir="rtl">

## 🎯 اهداف جلسه

در پایان این جلسه باید بتوانید:

- یک <code dir="ltr">DataFrame</code> را بر اساس یک یا چند ستون مرتب کنید و جای مقدارهای گمشده را در ترتیب پیش‌بینی کنید،
- با <code dir="ltr">rank</code>، <code dir="ltr">nlargest</code> و <code dir="ltr">nsmallest</code> بهترین‌ها را پیدا کنید و رفتار آن‌ها را در حالت **تساوی** بدانید،
- الگوی **تقسیم، اعمال، ترکیب** (split-apply-combine) را با <code dir="ltr">groupby</code> به کار ببرید و خروجی آن را قبل از اجرا پیش‌بینی کنید،
- تفاوت <code dir="ltr">agg</code>، <code dir="ltr">transform</code> و <code dir="ltr">filter</code> را بدانید و برای هر مسئله ابزار درست را انتخاب کنید،
- بدانید <code dir="ltr">apply</code> چه وقت لازم است و چه وقت باید به جای آن از عملیات **برداری** استفاده کرد،
- با <code dir="ltr">pivot&#95;table</code> و <code dir="ltr">crosstab</code> جدول‌های خلاصه بسازید و سطر و ستون <code dir="ltr">All</code> آن را درست تفسیر کنید.

> 📖 روش خواندن این جزوه و معنی علامت‌ها (💡 🔍 ❓ ⚠️ 📌 📝) در ابتدای جلسه‌ی ۱ توضیح داده شده است.

</div>

<div dir="rtl">

## فهرست مطالب

1. مرتب‌سازی
2. <code dir="ltr">groupby</code> و تجمیع
3. <code dir="ltr">apply</code>
4. <code dir="ltr">pivot&#95;table</code>

</div>

<div dir="rtl">

### داده‌های این جلسه

در این جلسه دو داده داریم:

- یک جدول کوچک فروش (<code dir="ltr">sales</code>) که خودمان می‌سازیم تا بتوانیم نتیجه‌ها را با دست هم حساب کنیم،
- داده‌ی واقعی **پنگوئن‌های پالمر** (<code dir="ltr">penguins.csv</code>) که در جلسه‌ی ۲۳ با آن آشنا شدید: ۳۴۴ پنگوئن از سه گونه (<code dir="ltr">Adelie</code>، <code dir="ltr">Chinstrap</code>، <code dir="ltr">Gentoo</code>) در سه جزیره، با اندازه‌ی نوک، طول باله، وزن و جنسیت.

سلول زیر داده را از پوشه‌ی <code dir="ltr">data</code> کنار نوت‌بوک می‌خواند و اگر فایل آنجا نبود، آن را از سایت درس دریافت می‌کند.

</div>

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

sales = pd.DataFrame({
    "store":   ["A", "B", "A", "C", "B", "A", "C", "B"],
    "product": ["pen", "pen", "book", "book", "bag", "bag", "pen", "book"],
    "units":   [10, 4, 3, 7, 2, 5, 8, 6],
    "price":   [1.5, 1.5, 12.0, 12.0, 25.0, 25.0, 1.5, 12.0],
})
sales["revenue"] = sales["units"] * sales["price"]
print(sales)

LOCAL = Path("data/penguins.csv")
URL = "https://haghbinh.github.io/Python-Course/Notebooks/data/penguins.csv"
penguins = pd.read_csv(LOCAL if LOCAL.exists() else URL)
print(penguins.shape)
print(penguins.isna().sum())

<div dir="rtl">

🔍 ستون <code dir="ltr">revenue</code> (درآمد) با یک عملیات **برداری** ساخته شد: ضرب دو ستون، سطر به سطر و بدون حلقه (جلسه‌ی ۲۳). در داده‌ی پنگوئن‌ها دو سطر اندازه‌گیری ندارند و ۱۱ پنگوئن جنسیت ثبت‌نشده دارند؛ این مقدارهای گمشده (<code dir="ltr">NaN</code>) را فعلاً نگه می‌داریم تا رفتار ابزارهای این جلسه را با آن‌ها ببینیم. روش‌های درست کار با داده‌ی گمشده موضوع جلسه‌ی ۲۶ است.

</div>

<div dir="rtl">

## ۱. مرتب‌سازی

### 💡 مفهوم

<code dir="ltr">df.sort&#95;values(by)</code> سطرها را بر اساس مقدار یک یا چند ستون مرتب می‌کند و یک <code dir="ltr">DataFrame</code> **جدید** برمی‌گرداند؛ مثل <code dir="ltr">sorted</code> در جلسه‌ی ۸ که لیست اصلی را دست نمی‌زد.

| پارامتر | کار |
|---|---|
| <code dir="ltr">by</code> | نام ستون، یا لیستی از نام‌ها برای مرتب‌سازی چندکلیدی |
| <code dir="ltr">ascending</code> | <code dir="ltr">True</code> (پیش‌فرض، صعودی) یا <code dir="ltr">False</code>؛ برای چند کلید، لیستی از <code dir="ltr">True</code>/<code dir="ltr">False</code> |
| <code dir="ltr">na&#95;position</code> | جای <code dir="ltr">NaN</code>: <code dir="ltr">"last"</code> (پیش‌فرض) یا <code dir="ltr">"first"</code> |

</div>

In [ ]:
print(sales.sort_values("revenue", ascending=False))
print(sales.sort_values(["store", "units"], ascending=[True, False]))

<div dir="rtl">

### 🔍 کد را بخوان

- مرتب‌سازی اول سطرها را از بیشترین درآمد (<code dir="ltr">125</code>) تا کمترین (<code dir="ltr">6</code>) می‌چیند.
- در مرتب‌سازی دوم، کلید **اول** <code dir="ltr">store</code> است (صعودی: <code dir="ltr">A</code>، <code dir="ltr">B</code>، <code dir="ltr">C</code>) و کلید **دوم** <code dir="ltr">units</code> فقط بین سطرهایی که <code dir="ltr">store</code> یکسان دارند تصمیم می‌گیرد (نزولی). این همان کاری است که در جلسه‌ی ۱۳ با <code dir="ltr">sorted(..., key=lambda r: (r[0], -r[1]))</code> انجام می‌دادید.
- به ستون **اندیس** (سمت چپ) دقت کنید: برچسب سطرها همراه سطر جابه‌جا می‌شود. سطری که برچسب <code dir="ltr">5</code> دارد، بعد از مرتب‌سازی هم برچسب <code dir="ltr">5</code> دارد.

### ⚠️ اشتباه رایج: <code dir="ltr">iloc</code> بعد از مرتب‌سازی

بعد از مرتب‌سازی، <code dir="ltr">loc[5]</code> همان سطر قبلی را می‌دهد (برچسب)، اما <code dir="ltr">iloc[0]</code> اولین سطر **ترتیب جدید** را (موقعیت). اگر شماره‌گذاری تازه از صفر می‌خواهید، <code dir="ltr">reset&#95;index(drop=True)</code> بزنید. برای برگرداندن ترتیب اولیه هم <code dir="ltr">sort&#95;index()</code> کافی است.

</div>

In [ ]:
top = sales.sort_values("revenue", ascending=False)
print(top.loc[5, "revenue"], top.iloc[0]["revenue"])
print(top.reset_index(drop=True).head(3))
print(top.sort_index().index.tolist())

<div dir="rtl">

### 💡 مقدارهای گمشده در مرتب‌سازی

<code dir="ltr">NaN</code> با هیچ عددی قابل مقایسه نیست؛ برای همین Pandas آن را **جدا** از بقیه می‌گذارد: به‌طور پیش‌فرض همیشه در **انتها**، چه صعودی مرتب کنید و چه نزولی.

</div>

In [ ]:
cols = ["species", "sex", "body_mass_g"]
print(penguins.sort_values("body_mass_g")[cols].tail(3))
print(penguins.sort_values("body_mass_g", ascending=False)[cols].tail(2))
print(penguins.sort_values("body_mass_g", na_position="first")[cols].head(3))

<div dir="rtl">

🔍 در هر دو جهت، دو پنگوئنی که وزن ندارند ته جدول‌اند؛ پس «سه سطر آخر مرتب‌سازی صعودی» لزوماً سنگین‌ترین‌ها **نیستند**. با <code dir="ltr">na&#95;position="first"</code> آن‌ها به اول جدول می‌آیند؛ راه ساده‌ای برای دیدن سطرهای ناقص.

### 💡 <code dir="ltr">nlargest</code>، <code dir="ltr">nsmallest</code> و <code dir="ltr">rank</code>

- <code dir="ltr">df.nlargest(k, col)</code> و <code dir="ltr">df.nsmallest(k, col)</code>: <code dir="ltr">k</code> سطر با بزرگ‌ترین / کوچک‌ترین مقدار؛ کوتاه‌تر و سریع‌تر از مرتب‌سازی کامل و برداشتن <code dir="ltr">head</code>، و <code dir="ltr">NaN</code> را خودبه‌خود کنار می‌گذارد.
- <code dir="ltr">s.rank()</code>: به هر مقدار **رتبه**‌اش را می‌دهد (از ۱). مهم‌ترین پارامترش <code dir="ltr">method</code> است که تکلیف **تساوی** را روشن می‌کند:

| <code dir="ltr">method</code> | مقدارهای برابر چه رتبه‌ای می‌گیرند؟ | مثال برای <code dir="ltr">[80, 80, 60]</code> نزولی |
|---|---|---|
| <code dir="ltr">"average"</code> (پیش‌فرض) | میانگین رتبه‌هایی که اشغال می‌کنند | <code dir="ltr">1.5, 1.5, 3</code> |
| <code dir="ltr">"min"</code> | کوچک‌ترین رتبه (روش مسابقات ورزشی) | <code dir="ltr">1, 1, 3</code> |
| <code dir="ltr">"max"</code> | بزرگ‌ترین رتبه | <code dir="ltr">2, 2, 3</code> |
| <code dir="ltr">"dense"</code> | کوچک‌ترین رتبه، بدون جای خالی بعد از تساوی | <code dir="ltr">1, 1, 2</code> |
| <code dir="ltr">"first"</code> | به ترتیب ظاهر شدن در داده | <code dir="ltr">1, 2, 3</code> |

</div>

In [ ]:
print(sales.nlargest(3, "revenue")[["store", "product", "revenue"]])
print(penguins.nsmallest(2, "body_mass_g")[cols])

scores = pd.Series([50, 80, 80, 60, 80, 40], index=list("abcdef"))
ranks = pd.DataFrame({"score": scores})
for m in ["average", "min", "dense", "first"]:
    ranks[m] = scores.rank(method=m, ascending=False)
print(ranks)

<div dir="rtl">

### 🔍 کد را بخوان

سه مقدار <code dir="ltr">80</code> (سطرهای <code dir="ltr">b</code>، <code dir="ltr">c</code> و <code dir="ltr">e</code>) رتبه‌های ۱ تا ۳ را اشغال می‌کنند:

- <code dir="ltr">average</code>: هر سه <code dir="ltr">2.0</code> (میانگین ۱، ۲ و ۳)؛ نفر بعدی (<code dir="ltr">60</code>) رتبه‌ی <code dir="ltr">4</code> می‌گیرد.
- <code dir="ltr">min</code>: هر سه <code dir="ltr">1</code>؛ نفر بعدی باز هم <code dir="ltr">4</code> (رتبه‌های ۲ و ۳ «خالی» می‌مانند).
- <code dir="ltr">dense</code>: هر سه <code dir="ltr">1</code>؛ نفر بعدی <code dir="ltr">2</code> (بدون جای خالی).
- <code dir="ltr">first</code>: <code dir="ltr">1</code>، <code dir="ltr">2</code>، <code dir="ltr">3</code> به ترتیب ظاهر شدن در داده؛ تساوی با ترتیب سطرها شکسته می‌شود.

خروجی <code dir="ltr">rank</code> همیشه از نوع <code dir="ltr">float</code> است، چون روش <code dir="ltr">average</code> ممکن است عدد اعشاری بدهد.

### ❓ پیش‌بینی کن

قبل از اجرا: خروجی هر خط چیست؟

</div>

In [ ]:
s = pd.Series([3, 1, 3, np.nan, 2])
print(s.rank().tolist())
print(s.rank(method="dense", ascending=False).tolist())
print(s.sort_values(ascending=False).index.tolist())

<div dir="rtl">

🔍 خط اول: <code dir="ltr">1</code> کوچک‌ترین است (رتبه‌ی ۱)، بعد <code dir="ltr">2</code> (رتبه‌ی ۲) و دو تا <code dir="ltr">3</code> رتبه‌های ۳ و ۴ را با میانگین <code dir="ltr">3.5</code> می‌گیرند: <code dir="ltr">[3.5, 1.0, 3.5, nan, 2.0]</code>. <code dir="ltr">NaN</code> رتبه نمی‌گیرد و <code dir="ltr">NaN</code> می‌ماند. خط دوم (نزولی و <code dir="ltr">dense</code>): <code dir="ltr">[1.0, 3.0, 1.0, nan, 2.0]</code>. خط سوم برچسب‌ها را به ترتیب نزولی مقدار می‌دهد و <code dir="ltr">NaN</code> (برچسب <code dir="ltr">3</code>) را در انتها می‌گذارد؛ ترتیب دو برچسب <code dir="ltr">0</code> و <code dir="ltr">2</code> که مقدار برابر دارند ممکن است در نسخه‌های مختلف فرق کند، چون مرتب‌سازی پیش‌فرض <code dir="ltr">sort&#95;values</code> برای مقدارهای برابر ترتیب اولیه را تضمین نمی‌کند (اگر لازم دارید، <code dir="ltr">kind="stable"</code> بدهید).

</div>

<div dir="rtl">

## ۲. <code dir="ltr">groupby</code> و تجمیع

### 💡 مفهوم: تقسیم، اعمال، ترکیب

در جلسه‌ی ۱۰ برای «جمع درآمد هر فروشگاه» یک دیکشنری می‌ساختید و روی سطرها حلقه می‌زدید. <code dir="ltr">groupby</code> همین کار را در سه مرحله و بدون حلقه انجام می‌دهد:

1. **تقسیم (split):** سطرها بر اساس مقدار کلید (اینجا <code dir="ltr">store</code>) به گروه‌ها تقسیم می‌شوند،
2. **اعمال (apply):** روی هر گروه یک تابع (مثلاً <code dir="ltr">sum</code>) اعمال می‌شود،
3. **ترکیب (combine):** نتیجه‌ها در یک <code dir="ltr">Series</code> یا <code dir="ltr">DataFrame</code> کنار هم گذاشته می‌شوند که **اندیس آن، کلیدهای گروه** است.

</div>

<p align="center"><img src="images/s24_split_apply_combine.png" width="850" alt="s24_split_apply_combine.png"></p>

<p align="center" dir="rtl"><i>الگوی تقسیم، اعمال، ترکیب برای <code dir="ltr">sales.groupby("store")["revenue"].sum()</code>.</i></p>

In [ ]:
# the S10 way: a dict and a loop
totals = {}
for store, rev in zip(sales["store"], sales["revenue"]):
    totals[store] = totals.get(store, 0) + rev
print(totals)

# the pandas way
print(sales.groupby("store")["revenue"].sum())

<div dir="rtl">

### 🔍 کد را بخوان

- <code dir="ltr">sales.groupby("store")</code> هنوز چیزی حساب نمی‌کند؛ فقط یک شیء «گروه‌بندی‌شده» می‌سازد (کمی شبیه تنبلی generatorها در جلسه‌ی ۱۴).
- <code dir="ltr">["revenue"]</code> ستونی را انتخاب می‌کند که می‌خواهیم خلاصه شود و <code dir="ltr">.sum()</code> مرحله‌ی اعمال است.
- نتیجه یک <code dir="ltr">Series</code> است که اندیسش <code dir="ltr">A</code>، <code dir="ltr">B</code> و <code dir="ltr">C</code> است؛ مرتب‌شده بر اساس کلید (پیش‌فرض <code dir="ltr">sort=True</code>).

### 💡 روی داده‌ی واقعی: <code dir="ltr">size</code> در برابر <code dir="ltr">count</code>

</div>

In [ ]:
print(penguins.groupby("species")["sex"].agg(["size", "count"]))

<div dir="rtl">

🔍 <code dir="ltr">size</code> تعداد **سطرهای** هر گروه است و <code dir="ltr">count</code> تعداد مقدارهای **غیرگمشده** در آن ستون. <code dir="ltr">Adelie</code> ۱۵۲ پنگوئن دارد، اما جنسیت فقط برای ۱۴۶ تا ثبت شده است. اگر می‌خواهید بدانید هر گروه چند عضو دارد، <code dir="ltr">size</code> بزنید؛ اگر می‌خواهید بدانید برای چند عضو داده دارید، <code dir="ltr">count</code>.

از اینجا به بعد، برای ساده ماندن مثال‌ها، سطرهای ناقص را کنار می‌گذاریم (این راه ساده اما همیشه درست نیست؛ جلسه‌ی ۲۶):

</div>

In [ ]:
penguins = penguins.dropna()
print(penguins.shape)
print(penguins.groupby("species")["body_mass_g"].mean().round(1))
print(penguins.groupby(["species", "sex"])["body_mass_g"].mean().round(0))

<div dir="rtl">

### 🔍 کد را بخوان

- ۳۳۳ سطر کامل باقی ماند.
- میانگین وزن <code dir="ltr">Gentoo</code> (حدود ۵۰۹۲ گرم) خیلی بیشتر از دو گونه‌ی دیگر (حدود ۳۷۰۰ گرم) است.
- با **دو کلید**، هر ترکیب <code dir="ltr">(species, sex)</code> یک گروه است و اندیس نتیجه **دوسطحی** (<code dir="ltr">MultiIndex</code>) می‌شود. در هر سه گونه، نرها سنگین‌ترند.

### 💡 چند خلاصه با هم: <code dir="ltr">agg</code>

<code dir="ltr">agg</code> (یا <code dir="ltr">aggregate</code>) اجازه می‌دهد چند تابع را یک‌جا اعمال کنید:

| شکل | نتیجه |
|---|---|
| <code dir="ltr">g["col"].agg(["mean", "max"])</code> | یک ستون، چند تابع |
| <code dir="ltr">g.agg({"col1": "mean", "col2": ["min", "max"]})</code> | برای هر ستون، تابع(های) خودش |
| <code dir="ltr">g.agg(new&#95;name=("col", "func"), ...)</code> | **تجمیع نام‌دار**: نام ستون‌های خروجی را خودتان تعیین می‌کنید |

</div>

In [ ]:
summary = penguins.groupby("species").agg(
    n=("body_mass_g", "size"),
    mass=("body_mass_g", "mean"),
    flipper=("flipper_length_mm", "median"),
)
print(summary.round(1))

print(penguins.groupby("species").agg({"bill_length_mm": "mean",
                                       "body_mass_g": ["min", "max"]}).round(1))

<div dir="rtl">

### 🔍 کد را بخوان

- تجمیع نام‌دار خواناترین شکل است: خروجی دقیقاً سه ستون <code dir="ltr">n</code>، <code dir="ltr">mass</code> و <code dir="ltr">flipper</code> دارد.
- شکل دیکشنری وقتی برای یک ستون چند تابع بدهید، ستون‌های **دوسطحی** می‌سازد (<code dir="ltr">body&#95;mass&#95;g</code> بالای <code dir="ltr">min</code> و <code dir="ltr">max</code>) که کار با آن‌ها سخت‌تر است. برای همین در کد واقعی معمولاً تجمیع نام‌دار ترجیح داده می‌شود.

### 💡 کلیدها به‌عنوان ستون: <code dir="ltr">as&#95;index=False</code> و <code dir="ltr">reset&#95;index</code>

نتیجه‌ی <code dir="ltr">groupby</code> کلیدها را در **اندیس** می‌گذارد. اگر کلیدها را به‌عنوان ستون معمولی لازم دارید (مثلاً برای ذخیره در CSV یا <code dir="ltr">merge</code> در جلسه‌ی ۲۵)، یکی از این دو راه:

</div>

In [ ]:
print(penguins.groupby("species", as_index=False)["body_mass_g"].mean().round(1))
print(penguins.groupby(["island", "species"]).size().reset_index(name="n"))

<div dir="rtl">

🔍 جدول دوم نشان می‌دهد <code dir="ltr">Adelie</code> تنها گونه‌ای است که در هر سه جزیره دیده شده؛ <code dir="ltr">Gentoo</code> فقط در <code dir="ltr">Biscoe</code> و <code dir="ltr">Chinstrap</code> فقط در <code dir="ltr">Dream</code>. ترکیب‌هایی که هیچ سطری ندارند، در خروجی <code dir="ltr">groupby</code> **اصلاً ظاهر نمی‌شوند** (نه با صفر).

### 💡 <code dir="ltr">transform</code>: نتیجه‌ی گروه برای هر سطر

<code dir="ltr">agg</code> هر گروه را به **یک عدد** خلاصه می‌کند (۳ گروه، ۳ سطر خروجی). <code dir="ltr">transform</code> نتیجه را به **همه‌ی سطرهای گروه** برمی‌گرداند؛ خروجی هم‌طول و هم‌اندیس با جدول اصلی است و می‌توان آن را مستقیم به‌عنوان ستون جدید گذاشت:

</div>

In [ ]:
mean_by_species = penguins.groupby("species")["body_mass_g"].transform("mean")
print(len(penguins), len(mean_by_species))

penguins["mass_vs_species"] = penguins["body_mass_g"] - mean_by_species
print(penguins[["species", "body_mass_g", "mass_vs_species"]].head(4).round(1))

<div dir="rtl">

### 🔍 کد را بخوان

برای هر پنگوئن، میانگین وزن **گونه‌ی خودش** کنار آن قرار گرفت و از وزنش کم شد. حالا <code dir="ltr">mass&#95;vs&#95;species</code> می‌گوید یک پنگوئن نسبت به هم‌گونه‌هایش سنگین است یا سبک؛ مقایسه‌ای که با وزن خام ممکن نیست، چون تقریباً هر <code dir="ltr">Gentoo</code> از هر <code dir="ltr">Adelie</code> سنگین‌تر است. این ایده (ویژگی **نسبت به گروه**) در ساختن ویژگی برای مدل‌های یادگیری ماشین بسیار رایج است (سؤال ۴).

### 💡 <code dir="ltr">filter</code>: نگه داشتن یا دور ریختن کل گروه

<code dir="ltr">g.filter(func)</code> تابع را روی هر گروه (یک <code dir="ltr">DataFrame</code> کوچک) صدا می‌زند و سطرهای گروه‌هایی را نگه می‌دارد که <code dir="ltr">func</code> برایشان <code dir="ltr">True</code> برگرداند:

</div>

In [ ]:
big_islands = penguins.groupby("island").filter(lambda g: len(g) > 100)
print(big_islands["island"].value_counts())

<div dir="rtl">

🔍 فقط جزیره‌هایی که بیش از ۱۰۰ پنگوئن (کامل) دارند باقی ماندند؛ <code dir="ltr">Torgersen</code> با ۴۷ پنگوئن حذف شد. خروجی <code dir="ltr">filter</code> سطرهای **اصلی** است، نه خلاصه.

| روش | تابع روی هر گروه چه برمی‌گرداند؟ | شکل خروجی |
|---|---|---|
| <code dir="ltr">agg</code> | یک عدد | یک سطر برای هر گروه |
| <code dir="ltr">transform</code> | یک عدد (یا یک ستون هم‌طول گروه) | هم‌شکل جدول اصلی |
| <code dir="ltr">filter</code> | <code dir="ltr">True</code> یا <code dir="ltr">False</code> | زیرمجموعه‌ای از سطرهای اصلی |

### ⚠️ اشتباه رایج: میانگین همه‌ی ستون‌ها

<code dir="ltr">penguins.groupby("species").mean()</code> خطا می‌دهد، چون ستون‌های متنی (<code dir="ltr">island</code>، <code dir="ltr">sex</code>) میانگین ندارند. یا ستون‌های عددی را صریحاً انتخاب کنید (<code dir="ltr">[["bill&#95;length&#95;mm", "body&#95;mass&#95;g"]]</code>) یا <code dir="ltr">mean(numeric&#95;only=True)</code> بنویسید.

### ❓ پیش‌بینی کن

</div>

In [ ]:
df = pd.DataFrame({"team": ["x", "y", "x", "y", "x"],
                   "score": [4, np.nan, 6, 3, 8]})
g = df.groupby("team")["score"]
print(g.agg(["size", "count", "sum", "mean"]))
print(g.transform("mean").tolist())

<div dir="rtl">

🔍 گروه <code dir="ltr">x</code> سه سطر با امتیازهای <code dir="ltr">4</code>، <code dir="ltr">6</code> و <code dir="ltr">8</code> دارد: <code dir="ltr">size</code> و <code dir="ltr">count</code> هر دو <code dir="ltr">3</code>، جمع <code dir="ltr">18</code> و میانگین <code dir="ltr">6</code>. گروه <code dir="ltr">y</code> دو سطر دارد اما یکی <code dir="ltr">NaN</code> است: <code dir="ltr">size</code> برابر <code dir="ltr">2</code>، <code dir="ltr">count</code> برابر <code dir="ltr">1</code>، جمع <code dir="ltr">3</code> و میانگین <code dir="ltr">3</code>؛ <code dir="ltr">NaN</code> در جمع و میانگین **نادیده** گرفته می‌شود و میانگین روی همان یک مقدار حساب می‌شود. <code dir="ltr">transform</code> میانگین هر گروه را به سطرهای همان گروه برمی‌گرداند: <code dir="ltr">[6.0, 3.0, 6.0, 3.0, 6.0]</code>.

</div>

<div dir="rtl">

## ۳. <code dir="ltr">apply</code>

### 💡 مفهوم

گاهی کاری که می‌خواهید با عملگرها و متدهای آماده‌ی Pandas انجام نمی‌شود و تابع خودتان را لازم دارید. سه ابزار:

| ابزار | تابع روی چه چیزی صدا زده می‌شود؟ |
|---|---|
| <code dir="ltr">s.map(dict&#95;or&#95;func)</code> | هر **عنصر** یک <code dir="ltr">Series</code>؛ با دیکشنری برای جایگزینی مقدارها |
| <code dir="ltr">s.apply(func)</code> | هر **عنصر** یک <code dir="ltr">Series</code> |
| <code dir="ltr">df.apply(func, axis=0)</code> | هر **ستون** (یک <code dir="ltr">Series</code>) |
| <code dir="ltr">df.apply(func, axis=1)</code> | هر **سطر** (یک <code dir="ltr">Series</code> که اندیسش نام ستون‌هاست) |

<code dir="ltr">lambda</code> (جلسه‌ی ۱۳) معمولاً همراه این ابزارها استفاده می‌شود.

</div>

In [ ]:
code_of = {"Adelie": 0, "Chinstrap": 1, "Gentoo": 2}
print(penguins["species"].map(code_of).value_counts())

print(penguins["body_mass_g"].apply(lambda g: g / 1000).head(3).tolist())

num = penguins[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]]
print(num.apply(lambda col: col.max() - col.min()))

ratio = penguins.apply(lambda row: row["bill_length_mm"] / row["bill_depth_mm"], axis=1)
print(ratio.head(3).round(2).tolist())

<div dir="rtl">

### 🔍 کد را بخوان

- <code dir="ltr">map</code> با دیکشنری، نام گونه را به کد عددی تبدیل کرد؛ مقدمه‌ی **کدگذاری متغیرهای رسته‌ای** در جلسه‌ی ۲۶. مقداری که در دیکشنری نیست، <code dir="ltr">NaN</code> می‌شود (سؤال ۶).
- <code dir="ltr">apply</code> با <code dir="ltr">axis=0</code> تابع را روی هر **ستون** اجرا کرد: دامنه‌ی تغییرات (بیشینه منهای کمینه) هر ویژگی. وزن دامنه‌ی ۳۶۰۰ دارد و عمق نوک فقط ۸٫۴؛ دلیل اینکه قبل از بسیاری از مدل‌ها ویژگی‌ها را **مقیاس‌بندی** می‌کنیم (جلسه‌ی ۲۶).
- با <code dir="ltr">axis=1</code> تابع روی هر **سطر** اجرا شد و نسبت طول به عمق نوک به دست آمد.

### ⚠️ اشتباه رایج: معنی <code dir="ltr">axis</code>

<code dir="ltr">axis=0</code> یعنی «در امتداد سطرها حرکت کن»، پس تابع **هر ستون** را می‌بیند؛ <code dir="ltr">axis=1</code> یعنی «در امتداد ستون‌ها»، پس تابع **هر سطر** را می‌بیند. همین قرارداد را در <code dir="ltr">mean(axis=...)</code> در NumPy (جلسه‌ی ۲۲) دیدید.

### 💡 <code dir="ltr">apply</code> سطری در برابر عملیات برداری

<code dir="ltr">apply(..., axis=1)</code> در واقع یک حلقه‌ی پایتونی روی سطرهاست و برای هر سطر یک <code dir="ltr">Series</code> جدید می‌سازد؛ کند است. همان نسبت را می‌توان با یک تقسیم **برداری** دو ستون حساب کرد که داخل NumPy و بدون حلقه‌ی پایتونی اجرا می‌شود:

</div>

In [ ]:
import time

big = pd.concat([penguins] * 100, ignore_index=True)
print(len(big))

t0 = time.perf_counter()
r1 = big.apply(lambda row: row["bill_length_mm"] / row["bill_depth_mm"], axis=1)
t1 = time.perf_counter()
r2 = big["bill_length_mm"] / big["bill_depth_mm"]
t2 = time.perf_counter()

print(np.allclose(r1, r2))
print(f"apply: {t1 - t0:.4f} s   vectorized: {t2 - t1:.4f} s")

<div dir="rtl">

🔍 هر دو روش نتیجه‌ی یکسان دارند (<code dir="ltr">True</code>)، اما روی ۳۳٬۳۰۰ سطر، نسخه‌ی برداری معمولاً **صدها برابر** سریع‌تر است (عدد دقیق به کامپیوتر شما بستگی دارد). <code dir="ltr">concat</code> که جدول را ۱۰۰ بار پشت سر هم گذاشت، موضوع جلسه‌ی ۲۵ است.

**قاعده:** اول دنبال راه برداری بگردید (عملگرهای حسابی روی ستون‌ها، متدهای <code dir="ltr">str</code> و <code dir="ltr">dt</code>، <code dir="ltr">np.where</code>، <code dir="ltr">groupby().transform</code>). <code dir="ltr">apply</code> را برای منطقی نگه دارید که واقعاً راه برداری ندارد.

### ❓ پیش‌بینی کن

دو سلول زیر یک ستون می‌سازند: «آیا این پنگوئن از میانگین گونه‌ی خودش سنگین‌تر است؟». آیا نتیجه‌ها یکسان است؟ کدام سریع‌تر است و چرا؟

</div>

In [ ]:
means = penguins.groupby("species")["body_mass_g"].mean()

heavy_1 = penguins.apply(lambda r: r["body_mass_g"] > means[r["species"]], axis=1)

group_mean = penguins.groupby("species")["body_mass_g"].transform("mean")
heavy_2 = penguins["body_mass_g"] > group_mean

print(heavy_1.equals(heavy_2), heavy_2.sum())

<div dir="rtl">

🔍 نتیجه‌ها یکسان‌اند (<code dir="ltr">True</code>) و ۱۵۸ پنگوئن از میانگین گونه‌ی خودشان سنگین‌ترند. نسخه‌ی اول برای هر سطر یک جست‌وجو در <code dir="ltr">means</code> انجام می‌دهد (حلقه‌ی پایتونی)؛ نسخه‌ی دوم با <code dir="ltr">transform</code> میانگین گروه را یک‌جا کنار هر سطر می‌گذارد و یک مقایسه‌ی برداری انجام می‌دهد؛ برای داده‌ی بزرگ خیلی سریع‌تر است.

</div>

<div dir="rtl">

## ۴. <code dir="ltr">pivot&#95;table</code>

### 💡 مفهوم

<code dir="ltr">pivot&#95;table</code> همان جدول محوری Excel است: مقدارهای یک ستون را بر اساس **دو** کلید خلاصه می‌کند، یکی در سطرها و دیگری در ستون‌ها.

| پارامتر | کار |
|---|---|
| <code dir="ltr">values</code> | ستونی که خلاصه می‌شود |
| <code dir="ltr">index</code> | کلید سطرها |
| <code dir="ltr">columns</code> | کلید ستون‌ها |
| <code dir="ltr">aggfunc</code> | تابع خلاصه‌سازی (پیش‌فرض <code dir="ltr">"mean"</code>) |
| <code dir="ltr">fill&#95;value</code> | مقدار خانه‌هایی که هیچ داده‌ای ندارند (پیش‌فرض <code dir="ltr">NaN</code>) |
| <code dir="ltr">margins</code> | اضافه کردن سطر و ستون <code dir="ltr">All</code> (خلاصه‌ی کل) |

</div>

In [ ]:
pt = sales.pivot_table(values="revenue", index="store", columns="product",
                       aggfunc="sum", fill_value=0, margins=True)
print(pt)

<div dir="rtl">

### 🔍 کد را بخوان

- هر خانه جمع درآمد یک فروشگاه از یک محصول است؛ مثلاً فروشگاه <code dir="ltr">A</code> از <code dir="ltr">bag</code> درآمد <code dir="ltr">125</code> داشته است.
- فروشگاه <code dir="ltr">C</code> اصلاً <code dir="ltr">bag</code> نفروخته؛ بدون <code dir="ltr">fill&#95;value=0</code> آن خانه <code dir="ltr">NaN</code> می‌شد.
- ستون <code dir="ltr">All</code> جمع هر سطر (درآمد کل هر فروشگاه: <code dir="ltr">176</code>، <code dir="ltr">128</code> و <code dir="ltr">96</code>، همان نتیجه‌ی بخش ۲) و سطر <code dir="ltr">All</code> جمع هر ستون است. خانه‌ی گوشه (<code dir="ltr">400</code>) درآمد کل است.

### 💡 همان جدول با <code dir="ltr">groupby</code>

<code dir="ltr">pivot&#95;table</code> در واقع یک <code dir="ltr">groupby</code> روی دو کلید است که کلید دوم از اندیس به ستون‌ها برده شده (<code dir="ltr">unstack</code>):

</div>

In [ ]:
print(sales.groupby(["store", "product"])["revenue"].sum().unstack(fill_value=0))

print(penguins.pivot_table(values="body_mass_g", index="species",
                           columns="sex", aggfunc="mean").round(0))

<div dir="rtl">

🔍 جدول اول همان <code dir="ltr">pt</code> بدون سطر و ستون <code dir="ltr">All</code> است. جدول دوم همان میانگین وزن گونه و جنسیت بخش ۲ است، اما در شکلی که مقایسه را ساده‌تر می‌کند: هر گونه یک سطر و دو جنس کنار هم.

### 💡 <code dir="ltr">crosstab</code>: جدول شمارش

برای شمردن تعداد ترکیب‌های دو ستون رسته‌ای، <code dir="ltr">pd.crosstab</code> کوتاه‌ترین راه است (معادل <code dir="ltr">pivot&#95;table</code> با <code dir="ltr">aggfunc="size"</code>):

</div>

In [ ]:
print(pd.crosstab(penguins["species"], penguins["island"], margins=True))
print(pd.crosstab(penguins["species"], penguins["sex"], normalize="index").round(2))

<div dir="rtl">

### 🔍 کد را بخوان

- جدول اول همان نتیجه‌ی <code dir="ltr">groupby(["island", "species"]).size()</code> است، اما خانه‌های خالی اینجا <code dir="ltr">0</code> دارند و یک نگاه کافی است تا ببینید کدام گونه کجا زندگی می‌کند.
- با <code dir="ltr">normalize="index"</code> هر سطر به کسرهایی تبدیل می‌شود که جمعشان ۱ است: در هر سه گونه، تعداد نر و ماده تقریباً برابر است (حدود ۵۰٪).

### ⚠️ اشتباه رایج: <code dir="ltr">All</code> در جدول میانگین

وقتی <code dir="ltr">aggfunc="mean"</code> است، خانه‌ی <code dir="ltr">All</code> میانگین **همه‌ی سطرهای خام** آن گروه است، نه میانگین خانه‌های جدول. اگر گروه‌ها اندازه‌ی متفاوت داشته باشند، این دو با هم فرق دارند (سؤال ۷).

### ❓ پیش‌بینی کن

بدون اجرا: جدول زیر چند سطر و چند ستون دارد؟ خانه‌ی <code dir="ltr">(A, pen)</code> و خانه‌ی <code dir="ltr">(C, bag)</code> چیست؟

</div>

In [ ]:
print(sales.pivot_table(values="units", index="store", columns="product", aggfunc="max"))

<div dir="rtl">

🔍 سه سطر (<code dir="ltr">A</code>، <code dir="ltr">B</code>، <code dir="ltr">C</code>) و سه ستون (<code dir="ltr">bag</code>، <code dir="ltr">book</code>، <code dir="ltr">pen</code>). خانه‌ی <code dir="ltr">(A, pen)</code> برابر <code dir="ltr">10</code> است و <code dir="ltr">(C, bag)</code> برابر <code dir="ltr">NaN</code>، چون <code dir="ltr">C</code> هرگز <code dir="ltr">bag</code> نفروخته و <code dir="ltr">fill&#95;value</code> داده نشده است. به خاطر وجود <code dir="ltr">NaN</code>، ستون <code dir="ltr">bag</code> از نوع <code dir="ltr">float</code> است (<code dir="ltr">5.0</code> و <code dir="ltr">2.0</code>).

</div>

<div dir="rtl">

## 📌 آنچه باید بلد باشید

- <code dir="ltr">sort&#95;values</code> با یک یا چند کلید و <code dir="ltr">ascending</code> به‌شکل لیست؛ <code dir="ltr">NaN</code> به‌طور پیش‌فرض همیشه در انتهاست (<code dir="ltr">na&#95;position</code>).
- برچسب سطرها با مرتب‌سازی جابه‌جا می‌شود: تفاوت <code dir="ltr">loc</code> و <code dir="ltr">iloc</code> بعد از مرتب‌سازی، <code dir="ltr">reset&#95;index(drop=True)</code> و <code dir="ltr">sort&#95;index</code>.
- <code dir="ltr">nlargest</code> / <code dir="ltr">nsmallest</code> و رفتار پنج روش <code dir="ltr">rank</code> در حالت تساوی.
- الگوی تقسیم، اعمال، ترکیب و معادل آن با حلقه و دیکشنری (جلسه‌ی ۱۰).
- تفاوت <code dir="ltr">size</code> و <code dir="ltr">count</code>؛ <code dir="ltr">groupby</code> با چند کلید و اندیس دوسطحی؛ <code dir="ltr">as&#95;index=False</code> و <code dir="ltr">reset&#95;index</code>.
- <code dir="ltr">agg</code> با لیست، دیکشنری و تجمیع نام‌دار.
- تفاوت <code dir="ltr">agg</code>، <code dir="ltr">transform</code> و <code dir="ltr">filter</code> از نظر شکل خروجی؛ ساختن ویژگی «نسبت به گروه» با <code dir="ltr">transform</code>.
- <code dir="ltr">map</code> و <code dir="ltr">apply</code> روی <code dir="ltr">Series</code>؛ معنی <code dir="ltr">axis=0</code> و <code dir="ltr">axis=1</code> در <code dir="ltr">DataFrame.apply</code>؛ چرا <code dir="ltr">apply</code> سطری کند است و جایگزین برداری آن.
- <code dir="ltr">pivot&#95;table</code> با <code dir="ltr">values</code>، <code dir="ltr">index</code>، <code dir="ltr">columns</code>، <code dir="ltr">aggfunc</code>، <code dir="ltr">fill&#95;value</code> و <code dir="ltr">margins</code>؛ معادل آن با <code dir="ltr">groupby</code> و <code dir="ltr">unstack</code>؛ <code dir="ltr">crosstab</code> و <code dir="ltr">normalize</code>.

</div>

<div dir="rtl">

## 📝 سؤالات مرور / نمونه سؤال امتحان

> اول خودتان جواب دهید (روی کاغذ، بدون اجرای کد)، بعد پاسخ را باز کنید و در آخر کد را اجرا کنید.

### سؤال ۱ — مرتب‌سازی و رتبه‌ی مدل‌ها

نتیجه‌ی آزمایش چند مدل روی داده‌ی اعتبارسنجی در جدول زیر آمده است (دقت <code dir="ltr">logreg</code> به خاطر خطا ثبت نشده):

</div>

```python
runs = pd.DataFrame({
    "model": ["knn", "tree", "svm", "logreg", "forest", "mlp"],
    "acc":   [0.91, 0.87, 0.91, np.nan, 0.93, 0.87],
    "time":  [2.0, 0.5, 8.0, 1.0, 5.0, 12.0],
})
```

<div dir="rtl">

الف) ترتیب ستون <code dir="ltr">model</code> در خروجی <code dir="ltr">runs.sort&#95;values(["acc", "time"], ascending=[False, True])</code> چیست؟ این مرتب‌سازی چه سیاستی را برای انتخاب مدل پیاده می‌کند؟

ب) در <code dir="ltr">runs.sort&#95;values("acc", na&#95;position="first")</code> اولین مدل کدام است؟

ج) خروجی <code dir="ltr">runs["acc"].rank(method="min", ascending=False)</code> و <code dir="ltr">runs["acc"].rank(method="dense", ascending=False)</code> را بنویسید.

د) <code dir="ltr">runs.nlargest(2, "acc")</code> چه سطرهایی برمی‌گرداند؟ با <code dir="ltr">keep="all"</code> چطور؟ چرا؟

<details>
<summary><b>پاسخ</b></summary>

الف) <code dir="ltr">forest, knn, svm, tree, mlp, logreg</code>. اول بر اساس دقت (نزولی)؛ بین مدل‌های با دقت برابر، مدل **سریع‌تر** جلو می‌آید (<code dir="ltr">knn</code> با ۲ ثانیه قبل از <code dir="ltr">svm</code> با ۸ ثانیه، و <code dir="ltr">tree</code> قبل از <code dir="ltr">mlp</code>). <code dir="ltr">logreg</code> که دقت ندارد، در انتهاست. سیاست: «بیشترین دقت؛ در تساوی، کمترین زمان».

ب) <code dir="ltr">logreg</code>.

ج) <code dir="ltr">min</code>: <code dir="ltr">[2.0, 4.0, 2.0, nan, 1.0, 4.0]</code>. <code dir="ltr">dense</code>: <code dir="ltr">[2.0, 3.0, 2.0, nan, 1.0, 3.0]</code>. با <code dir="ltr">min</code>، بعد از دو مدل هم‌رتبه‌ی دوم، رتبه‌ی بعدی <code dir="ltr">4</code> است؛ با <code dir="ltr">dense</code>، <code dir="ltr">3</code>.

د) <code dir="ltr">forest</code> و <code dir="ltr">knn</code>. دو مدل <code dir="ltr">knn</code> و <code dir="ltr">svm</code> برای جایگاه دوم مساوی‌اند و <code dir="ltr">nlargest</code> به‌طور پیش‌فرض (<code dir="ltr">keep="first"</code>) اولی را به ترتیب داده نگه می‌دارد. با <code dir="ltr">keep="all"</code> هر سه سطر (<code dir="ltr">forest</code>، <code dir="ltr">knn</code>، <code dir="ltr">svm</code>) برمی‌گردند، حتی اگر بیش از <code dir="ltr">k</code> سطر شود.

</details>

### سؤال ۲ — ردگیری تقسیم، اعمال، ترکیب

جدول زیر را در نظر بگیرید:

</div>

```python
t = pd.DataFrame({
    "city":  ["Bushehr", "Shiraz", "Bushehr", "Shiraz", "Yazd", "Bushehr"],
    "temp":  [31, 24, np.nan, 26, 35, 33],
    "humid": [70, 30, 75, 35, 15, 65],
})
```

<div dir="rtl">

بدون اجرا، خروجی هر عبارت را بنویسید:

الف) <code dir="ltr">t.groupby("city")["temp"].agg(["size", "count", "mean"])</code>

ب) <code dir="ltr">t.groupby("city")["humid"].transform("max").tolist()</code>

ج) <code dir="ltr">t.groupby("city").filter(lambda g: g["temp"].notna().all())["city"].unique()</code>

د) <code dir="ltr">t.groupby("city").agg(hot=("temp", "max"), dry=("humid", "min")).loc["Shiraz"]</code>

ه) همین نتیجه‌ی بند (الف) را فقط با یک حلقه و دیکشنری (بدون Pandas، با داده به شکل لیست تاپل‌ها) بنویسید. کدام قسمت کد شما «تقسیم»، کدام «اعمال» و کدام «ترکیب» است؟

<details>
<summary><b>پاسخ</b></summary>

الف)

| city | size | count | mean |
|---|---|---|---|
| Bushehr | 3 | 2 | 32.0 |
| Shiraz | 2 | 2 | 25.0 |
| Yazd | 1 | 1 | 35.0 |

میانگین <code dir="ltr">Bushehr</code> فقط روی دو مقدار موجود (<code dir="ltr">31</code> و <code dir="ltr">33</code>) حساب می‌شود.

ب) <code dir="ltr">[75, 35, 75, 35, 15, 75]</code>: بیشینه‌ی رطوبت هر شهر، برای هر سطرِ آن شهر.

ج) <code dir="ltr">['Shiraz', 'Yazd']</code>: گروه <code dir="ltr">Bushehr</code> یک دمای گمشده دارد و کل گروهش حذف می‌شود.

د) یک <code dir="ltr">Series</code> با <code dir="ltr">hot = 26.0</code> و <code dir="ltr">dry = 30.0</code>. (دقت کنید <code dir="ltr">hot</code> اعشاری است، چون ستون <code dir="ltr">temp</code> به خاطر <code dir="ltr">NaN</code> از نوع <code dir="ltr">float</code> است.)

ه)

<div dir="ltr">

```python
rows = [("Bushehr", 31), ("Shiraz", 24), ("Bushehr", None),
        ("Shiraz", 26), ("Yazd", 35), ("Bushehr", 33)]
groups = {}
for city, temp in rows:                  # split
    groups.setdefault(city, []).append(temp)
result = {}
for city, temps in groups.items():       # apply
    present = [x for x in temps if x is not None]
    result[city] = (len(temps), len(present), sum(present) / len(present))
print(result)                            # combine: one entry per group
```

</div>

حلقه‌ی اول سطرها را بر اساس کلید در لیست‌ها **تقسیم** می‌کند، حلقه‌ی دوم روی هر گروه سه خلاصه را **اعمال** می‌کند و دیکشنری <code dir="ltr">result</code> نتیجه‌ها را با کلید گروه **ترکیب** می‌کند. (<code dir="ltr">setdefault</code> یک متد دیکشنری است که اگر کلید نباشد، مقدار پیش‌فرض را می‌گذارد و برمی‌گرداند؛ می‌توانستید از <code dir="ltr">defaultdict(list)</code> جلسه‌ی ۱۷ هم استفاده کنید.)

</details>

### سؤال ۳ — <code dir="ltr">k</code> تای برتر در هر گروه

می‌خواهیم دو پنگوئن سنگین هر گونه را پیدا کنیم. دو راه‌حل:

</div>

```python
way1 = (penguins.sort_values("body_mass_g", ascending=False)
            .groupby("species").head(2))
way2 = penguins.groupby("species")["body_mass_g"].nlargest(2)
```

<div dir="rtl">

الف) هر کدام چند سطر برمی‌گرداند؟ ترتیب سطرها و نوع خروجی دو راه چه فرقی دارد؟

ب) خروجی <code dir="ltr">way1[["species", "body&#95;mass&#95;g"]]</code> را بنویسید (وزن‌ها: <code dir="ltr">Gentoo</code> با ۶۳۰۰ و ۶۰۵۰، <code dir="ltr">Chinstrap</code> با ۴۸۰۰ و ۴۵۵۰، <code dir="ltr">Adelie</code> با ۴۷۷۵ و ۴۷۲۵ گرم).

ج) اگر در <code dir="ltr">way1</code> جای <code dir="ltr">sort&#95;values</code> و <code dir="ltr">groupby</code> را عوض کنیم (<code dir="ltr">penguins.groupby("species").head(2).sort&#95;values(...)</code>) چه چیزی به دست می‌آید؟

د) در توصیه‌گرها (recommender systems) «<code dir="ltr">k</code> آیتم با بیشترین امتیاز برای هر کاربر» دقیقاً همین مسئله است. اگر بخواهیم در حالت تساوی امتیاز، آیتم ارزان‌تر جلو بیاید، <code dir="ltr">way1</code> را چطور تغییر می‌دهید؟

<details>
<summary><b>پاسخ</b></summary>

الف) هر دو ۶ مقدار (۲ برای هر یک از ۳ گونه). <code dir="ltr">way1</code> یک <code dir="ltr">DataFrame</code> با همه‌ی ستون‌هاست و سطرها به ترتیب **وزن کلی** آمده‌اند (گونه‌ها در هم)؛ <code dir="ltr">way2</code> یک <code dir="ltr">Series</code> فقط از وزن است با اندیس دوسطحی <code dir="ltr">(species, شماره‌ی سطر اصلی)</code> و به ترتیب گونه گروه‌بندی شده است.

ب)

| species | body_mass_g |
|---|---|
| Gentoo | 6300.0 |
| Gentoo | 6050.0 |
| Chinstrap | 4800.0 |
| Adelie | 4775.0 |
| Adelie | 4725.0 |
| Chinstrap | 4550.0 |

<code dir="ltr">groupby().head(2)</code> بعد از مرتب‌سازی، دو سطر **اول** هر گروه را با حفظ ترتیب فعلی برمی‌دارد؛ برای همین گونه‌ها یک در میان آمده‌اند.

ج) دو سطر **اول هر گونه به ترتیب داده‌ی خام** (نه سنگین‌ترین‌ها) و بعد مرتب‌سازی همان ۶ سطر. ترتیب عملیات مهم است: اول مرتب‌سازی، بعد برداشتن <code dir="ltr">head</code>.

د) مرتب‌سازی دوکلیدی: <code dir="ltr">sort&#95;values(["score", "price"], ascending=[False, True])</code> و بعد <code dir="ltr">groupby("user").head(k)</code>.

</details>

### سؤال ۴ — استانداردسازی درون‌گروهی

می‌خواهیم برای هر پنگوئن یک ویژگی «سنگینی نسبت به گونه» بسازیم:

</div>

```python
g = penguins.groupby("species")["body_mass_g"]
penguins["z_mass"] = (penguins["body_mass_g"] - g.transform("mean")) / g.transform("std")
```

<div dir="rtl">

الف) چرا اینجا <code dir="ltr">transform</code> لازم است و <code dir="ltr">agg</code> کار نمی‌کند؟

ب) میانگین و انحراف معیار ستون <code dir="ltr">z&#95;mass</code> **در هر گونه** چقدر است؟ چرا؟

ج) یک <code dir="ltr">Adelie</code> با وزن ۴۷۰۰ گرم و یک <code dir="ltr">Gentoo</code> با همان وزن، <code dir="ltr">z&#95;mass</code> تقریباً <code dir="ltr">2.17</code> و <code dir="ltr">-0.78</code> دارند. هر کدام را تفسیر کنید. چرا این ویژگی برای مدلی که می‌خواهد پنگوئن‌های «غیرعادی» را پیدا کند از وزن خام مفیدتر است؟

د) این کار را روی **کل** داده انجام دادیم. اگر قرار باشد بعداً داده را به آموزش و آزمون تقسیم کنیم، چه اشکالی پیش می‌آید و راه درست چیست؟

<details>
<summary><b>پاسخ</b></summary>

الف) <code dir="ltr">agg</code> برای هر گونه یک عدد برمی‌گرداند (۳ مقدار)، اما ما برای هر سطر (۳۳۳ مقدار) میانگین و انحراف معیار **گونه‌ی خودش** را لازم داریم تا از وزنش کم و بر آن تقسیم کنیم. <code dir="ltr">transform</code> همان خلاصه‌ها را هم‌اندیس با جدول اصلی برمی‌گرداند.

ب) میانگین <code dir="ltr">0</code> و انحراف معیار <code dir="ltr">1</code> (با خطای گرد کردن بسیار کوچک)؛ چون در هر گروه، از هر مقدار میانگین **همان گروه** کم و بر انحراف معیار **همان گروه** تقسیم شده است؛ همان استانداردسازی جلسه‌ی ۲۲، این بار جداگانه در هر گروه.

ج) آن <code dir="ltr">Adelie</code> حدود ۲٫۲ انحراف معیار از میانگین گونه‌اش سنگین‌تر است: برای یک <code dir="ltr">Adelie</code> بسیار سنگین. آن <code dir="ltr">Gentoo</code> حدود ۰٫۸ انحراف معیار سبک‌تر از میانگین <code dir="ltr">Gentoo</code>هاست: کمی سبک، کاملاً معمولی. با وزن خام هر دو یکسان‌اند و هیچ‌کدام غیرعادی به نظر نمی‌رسند؛ <code dir="ltr">z&#95;mass</code> اثر گونه را حذف می‌کند و فقط «فاصله از حالت عادی گروه» را نشان می‌دهد.

د) میانگین و انحراف معیار هر گونه با استفاده از داده‌ی **آزمون** هم حساب شده و اطلاعات آزمون به ویژگی‌های آموزش راه پیدا کرده است (**نشت داده**، جلسه‌ی ۹). راه درست: آماره‌های هر گروه را **فقط روی داده‌ی آموزش** حساب کنید (<code dir="ltr">train.groupby("species")["body&#95;mass&#95;g"].agg(["mean", "std"])</code>) و همان اعداد را برای تبدیل آموزش و آزمون به کار ببرید؛ همان قاعده‌ی «<code dir="ltr">fit</code> فقط روی آموزش» در کلاس <code dir="ltr">StandardScaler</code> جلسه‌ی ۱۸ و در جلسه‌ی ۲۶.

</details>

### سؤال ۵ — کدگذاری با میانگین هدف (target encoding)

یک روش رایج برای تبدیل یک ستون رسته‌ای به عدد، جایگزین کردن هر دسته با **میانگین متغیر هدف** در آن دسته است. داده‌ی کوچک زیر قیمت (هدف) چند آپارتمان را نشان می‌دهد:

</div>

```python
h = pd.DataFrame({
    "city":  ["Tehran", "Tehran", "Tehran", "Shiraz", "Shiraz", "Yazd"],
    "price": [10, 12, 14, 7, 9, 20],
})
g = h.groupby("city")["price"]
h["enc_full"] = g.transform("mean")
h["enc_loo"] = (g.transform("sum") - h["price"]) / (g.transform("count") - 1)
```

<div dir="rtl">

الف) ستون‌های <code dir="ltr">enc&#95;full</code> و <code dir="ltr">enc&#95;loo</code> را بنویسید.

ب) مدلی که فقط <code dir="ltr">enc&#95;full</code> را به‌عنوان ورودی ببیند، قیمت آپارتمان <code dir="ltr">Yazd</code> را **دقیقاً** پیش‌بینی می‌کند. چرا این یک موفقیت واقعی نیست؟ این مشکل برای دسته‌های کوچک بدتر است یا بزرگ؟

ج) <code dir="ltr">enc&#95;loo</code> (leave-one-out) چه چیزی را درست می‌کند؟ چرا برای <code dir="ltr">Yazd</code> مقدار <code dir="ltr">NaN</code> شده و با آن چه می‌کنید؟

د) در عمل، میانگین‌ها را باید روی چه داده‌ای حساب کرد تا روی داده‌ی آزمون نشت رخ ندهد؟ برای شهری که در آزمون هست ولی در آموزش نیست، چه مقداری می‌گذارید؟

<details>
<summary><b>پاسخ</b></summary>

الف) <code dir="ltr">enc&#95;full</code>: <code dir="ltr">[12, 12, 12, 8, 8, 20]</code> (میانگین قیمت هر شهر). <code dir="ltr">enc&#95;loo</code>: <code dir="ltr">[13, 12, 11, 9, 7, NaN]</code>؛ مثلاً برای اولین سطر تهران: <code dir="ltr">(12 + 14) / 2 = 13</code>.

ب) چون <code dir="ltr">Yazd</code> فقط یک آپارتمان دارد، میانگین آن دسته **همان قیمت خودش** است: ورودی مدل، جواب را در خود دارد (**نشت هدف**). روی آپارتمان جدیدی در یزد، این دقت کامل تکرار نمی‌شود. مشکل برای دسته‌های **کوچک** بدتر است؛ در دسته‌های بزرگ، سهم یک سطر در میانگین ناچیز است.

ج) برای هر سطر، میانگین **بقیه‌ی** سطرهای همان دسته را می‌گذارد؛ پس قیمت خود سطر در ویژگی‌اش اثری ندارد. برای <code dir="ltr">Yazd</code> «بقیه» وجود ندارد (<code dir="ltr">count - 1 = 0</code>) و تقسیم بر صفر در Pandas <code dir="ltr">NaN</code> می‌دهد. معمولاً این‌ها را با میانگین **کل** هدف پر می‌کنند (یا میانگین دسته را به سمت میانگین کل «هموار» می‌کنند، شبیه هموارسازی لاپلاس).

د) فقط روی داده‌ی **آموزش**: <code dir="ltr">means = train.groupby("city")["price"].mean()</code> و بعد <code dir="ltr">test["city"].map(means)</code>. شهر ناشناخته با <code dir="ltr">map</code> به <code dir="ltr">NaN</code> تبدیل می‌شود و باید آن را با میانگین کل قیمت آموزش پر کرد (<code dir="ltr">fillna(train["price"].mean())</code>).

</details>

### سؤال ۶ — <code dir="ltr">apply</code>، <code dir="ltr">map</code> و جایگزین برداری

الف) خروجی کد زیر چیست؟ چرا؟

</div>

```python
codes = penguins["species"].map({"Adelie": 0, "Gentoo": 1})
print(codes.isna().sum())
```

<div dir="rtl">

ب) کد زیر را بدون <code dir="ltr">apply</code> و فقط با عملیات برداری بازنویسی کنید:

</div>

```python
def size_label(row):
    if row["body_mass_g"] >= 5000:
        return "large"
    elif row["body_mass_g"] >= 3500:
        return "medium"
    return "small"

penguins["size"] = penguins.apply(size_label, axis=1)
```

<div dir="rtl">

ج) کد زیر تلاش می‌کند هر ستون عددی را بین صفر و یک ببرد (min-max). کدام <code dir="ltr">axis</code> درست است و خروجی آن برای ستون <code dir="ltr">[2, 4, 6]</code> چیست؟ نسخه‌ی برداری آن (بدون <code dir="ltr">apply</code>) را بنویسید.

</div>

```python
num = penguins[["bill_length_mm", "body_mass_g"]]
scaled = num.apply(lambda col: (col - col.min()) / (col.max() - col.min()), axis=____)
```

<div dir="rtl">

<details>
<summary><b>پاسخ</b></summary>

الف) <code dir="ltr">68</code>: همه‌ی پنگوئن‌های <code dir="ltr">Chinstrap</code> (۶۸ تا در داده‌ی کامل). <code dir="ltr">map</code> با دیکشنری، هر مقداری را که در دیکشنری **نیست** به <code dir="ltr">NaN</code> تبدیل می‌کند؛ بدون خطا و بی‌صدا. (اگر می‌خواهید مقدارهای ناموجود دست‌نخورده بمانند، <code dir="ltr">replace</code> به جای <code dir="ltr">map</code> استفاده کنید.)

ب) با <code dir="ltr">np.select</code> (چند شرط، به ترتیب بررسی می‌شوند؛ مثل <code dir="ltr">if</code> / <code dir="ltr">elif</code>):

<div dir="ltr">

```python
m = penguins["body_mass_g"]
penguins["size"] = np.select([m >= 5000, m >= 3500], ["large", "medium"], default="small")
```

</div>

یا با <code dir="ltr">pd.cut(m, bins=[0, 3500, 5000, np.inf], right=False, labels=["small", "medium", "large"])</code>. هر دو یک بار روی کل ستون اجرا می‌شوند، نه یک بار برای هر سطر.

ج) <code dir="ltr">axis=0</code> (تابع باید هر **ستون** را ببیند تا کمینه و بیشینه‌ی همان ستون را حساب کند). برای <code dir="ltr">[2, 4, 6]</code> نتیجه <code dir="ltr">[0.0, 0.5, 1.0]</code> است. نسخه‌ی برداری: <code dir="ltr">(num - num.min()) / (num.max() - num.min())</code>؛ <code dir="ltr">num.min()</code> یک <code dir="ltr">Series</code> با یک مقدار برای هر ستون است و تفریق، آن را ستون به ستون روی جدول **broadcast** می‌کند (جلسه‌ی ۲۲).

</details>

### سؤال ۷ — <code dir="ltr">pivot&#95;table</code>، <code dir="ltr">groupby</code> و حاشیه‌ها

</div>

```python
pt = sales.pivot_table(values="units", index="store", columns="product",
                       aggfunc="mean", margins=True)
```

<div dir="rtl">

الف) جدول <code dir="ltr">pt</code> را با دست کامل کنید (همه‌ی خانه‌ها، از جمله سطر و ستون <code dir="ltr">All</code>).

ب) آیا خانه‌ی <code dir="ltr">(All, All)</code> با میانگین سه عدد ستون <code dir="ltr">All</code> (برای <code dir="ltr">A</code>، <code dir="ltr">B</code> و <code dir="ltr">C</code>) برابر است؟ چرا؟

ج) همین جدول (بدون حاشیه‌ها) را با <code dir="ltr">groupby</code> و <code dir="ltr">unstack</code> بنویسید.

د) اگر <code dir="ltr">aggfunc="sum"</code> بود، آیا جواب بند (ب) فرق می‌کرد؟

<details>
<summary><b>پاسخ</b></summary>

الف)

| store | bag | book | pen | All |
|---|---|---|---|---|
| A | 5.0 | 3.000 | 10.000 | 6.000 |
| B | 2.0 | 6.000 | 4.000 | 4.000 |
| C | NaN | 7.000 | 8.000 | 7.500 |
| All | 3.5 | 5.333 | 7.333 | 5.625 |

مثلاً ستون <code dir="ltr">All</code> برای <code dir="ltr">A</code> میانگین سه فروش <code dir="ltr">A</code> است: <code dir="ltr">(10 + 3 + 5) / 3 = 6</code>؛ و <code dir="ltr">(All, All)</code> میانگین همه‌ی ۸ فروش است: <code dir="ltr">45 / 8 = 5.625</code>.

ب) خیر. میانگین سه عدد <code dir="ltr">6</code>، <code dir="ltr">4</code> و <code dir="ltr">7.5</code> برابر <code dir="ltr">5.833</code> است. خانه‌ی <code dir="ltr">(All, All)</code> میانگین **همه‌ی سطرهای خام** است و فروشگاه‌هایی که سطر بیشتری دارند (<code dir="ltr">A</code> و <code dir="ltr">B</code> با ۳ سطر در برابر <code dir="ltr">C</code> با ۲ سطر) وزن بیشتری دارند. میانگینِ میانگین‌ها فقط وقتی با میانگین کل برابر است که گروه‌ها هم‌اندازه باشند.

ج) <code dir="ltr">sales.groupby(["store", "product"])["units"].mean().unstack()</code>

د) بله: جمع‌ها جمع‌پذیرند؛ جمع سه عدد ستون <code dir="ltr">All</code> (<code dir="ltr">18 + 12 + 15 = 45</code>) دقیقاً با <code dir="ltr">(All, All)</code> برابر است. این ویژگی برای میانگین، میانه و بیشتر آماره‌های دیگر برقرار نیست؛ همیشه مشخص کنید حاشیه‌ی جدول با کدام تابع و روی چه داده‌ای حساب شده است.

</details>

</div>

<div dir="rtl">

## خلاصه‌ی جلسه

- <code dir="ltr">sort&#95;values</code> یک یا چند کلید و جهت هر کدام را می‌گیرد؛ <code dir="ltr">NaN</code> جدا (پیش‌فرض در انتها) قرار می‌گیرد و برچسب سطرها همراه سطر جابه‌جا می‌شود.
- <code dir="ltr">rank</code> پنج روش برای تساوی دارد؛ <code dir="ltr">nlargest</code> و <code dir="ltr">nsmallest</code> راه کوتاه پیدا کردن بهترین‌ها هستند.
- <code dir="ltr">groupby</code> الگوی تقسیم، اعمال، ترکیب است: <code dir="ltr">agg</code> هر گروه را خلاصه می‌کند، <code dir="ltr">transform</code> نتیجه‌ی گروه را به سطرها برمی‌گرداند و <code dir="ltr">filter</code> کل گروه را نگه می‌دارد یا حذف می‌کند.
- <code dir="ltr">apply</code> انعطاف‌پذیر اما کند است؛ اول دنبال راه برداری بگردید.
- <code dir="ltr">pivot&#95;table</code> همان <code dir="ltr">groupby</code> روی دو کلید است که یک کلید را به ستون‌ها برده؛ حاشیه‌ی <code dir="ltr">All</code> را با دقت تفسیر کنید.
- آماره‌های گروه (میانگین، انحراف معیار، میانگین هدف) را فقط روی داده‌ی آموزش حساب کنید تا نشت داده رخ ندهد.

**جلسه‌ی بعد:** Pandas ۳؛ چسباندن جدول‌ها به هم با <code dir="ltr">concat</code>، ترکیب جدول‌ها با <code dir="ltr">merge</code> و <code dir="ltr">join</code>، و کار با داده‌های زمانی.

</div>